<a href="https://colab.research.google.com/github/nivyanaboruah/FlyRank/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nivyanaboruah/FlyRank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I selected Random Forest Classifier to predict whether a content page is declining based on its historical performance and content characteristics.

Random Forest is suitable because the dataset contains numerical and categorical features, and the relationship between content characteristics and performance may be nonlinear. It can capture feature interactions and provides feature importance for interpretation.

I will compare it against a simple baseline model using the same validation split and evaluation metrics. The objective is to identify a model that improves prediction without unnecessary complexity.

In [ ]:
import pandas as pd
import numpy as np

# Load your FlyRank dataset
url = "https://raw.githubusercontent.com/nivyanaboruah/FlyRank/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-grouped train/test split because multiple content pages may belong to the same client. A random row-level split could place pages from the same client in both training and testing data, causing data leakage.

I will use client_id only for grouping and not as a model feature. This split evaluates whether the model can generalize to clients it has not seen during training.

The target is is_declining_label. I will exclude trend_direction and trend_pct because they are directly used to construct the target and would cause target leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# Load your dataset if not already loaded
# df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# Safe feature columns
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "age_tier_order"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier"
]

feature_columns = numeric_features + categorical_features

# Keep only available columns
feature_columns = [
    col for col in feature_columns
    if col in df.columns
]

# Prepare X and y
X = df[feature_columns].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

# Remove rows with missing target or group
valid_rows = y.notna() & groups.notna()

X = X.loc[valid_rows]
y = y.loc[valid_rows]
groups = groups.loc[valid_rows]

# Grouped train/test split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print(
    "Train clients:",
    groups.iloc[train_idx].nunique()
)

print(
    "Test clients:",
    groups.iloc[test_idx].nunique()
)

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Baseline model
baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

# Random Forest
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# Evaluation
def evaluate_model(name, y_true, y_pred, model=None, X_data=None):
    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        )
    }

    if model is not None and X_data is not None:
        probabilities = model.predict_proba(X_data)[:, 1]

        results["ROC-AUC"] = roc_auc_score(
            y_true, probabilities
        )
    else:
        results["ROC-AUC"] = np.nan

    return results

comparison = pd.DataFrame([
    evaluate_model(
        "Baseline",
        y_test,
        baseline_pred
    ),
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred,
        rf_model,
        X_test
    )
])

display(comparison.round(3))

print(classification_report(
    y_test,
    rf_pred,
    zero_division=0
))


Training rows: 23837
Testing rows: 6163
Train clients: 25
Test clients: 7


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Baseline,0.511,0.511,1.000,0.676,NaN
1,Random Forest,0.530,0.536,0.607,0.569,0.53


              precision    recall  f1-score   support

           0       0.52      0.45      0.48      3014
           1       0.54      0.61      0.57      3149

    accuracy                           0.53      6163
   macro avg       0.53      0.53      0.53      6163
weighted avg       0.53      0.53      0.53      6163



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Identify incorrect predictions
wrong_mask = rf_pred != y_test

errors = X_test.loc[wrong_mask].copy()

errors["actual"] = y_test.loc[wrong_mask]
errors["predicted"] = rf_pred[wrong_mask]

errors["error_type"] = np.where(
    errors["actual"] == 1,
    "False negative",
    "False positive"
)

print("Total incorrect predictions:", len(errors))

display(errors.head(10))



Total incorrect predictions: 2894


,search_volume,competition,cpc,word_count,char_count,content_age_days,days_since_last_update,age_tier_order,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,char_count_tier,actual,predicted,error_type
1,90.0,0.01,0.05,2481.0,15562.0,445,25,6,LOW,keyword article,informational,365+,0-30,2000-3500,15000-25000,1,0,False negative
13,10.0,0.00,0.00,1342.0,8469.0,238,103,5,LOW,keyword article,informational,181-365,91-180,1000-2000,8000-15000,0,1,False positive
23,0.0,0.00,0.00,NaN,NaN,502,20,6,LOW,keyword article,informational,365+,0-30,NaN,NaN,1,0,False negative
26,0.0,0.00,0.00,2686.0,17181.0,300,13,5,LOW,keyword article,informational,181-365,0-30,2000-3500,15000-25000,0,1,False positive
36,0.0,0.00,0.00,2510.0,15518.0,187,20,5,LOW,keyword article,informational,181-365,0-30,2000-3500,15000-25000,0,1,False positive
39,90.0,0.06,0.03,3666.0,21824.0,348,104,5,LOW,keyword article,informational,181-365,91-180,3500+,15000-25000,1,0,False negative
54,0.0,0.00,0.00,NaN,NaN,502,22,6,LOW,keyword article,informational,365+,0-30,NaN,NaN,1,0,False negative
56,0.0,0.00,0.00,3158.0,18868.0,145,20,4,LOW,keyword article,informational,91-180,0-30,2000-3500,15000-25000,0,1,False positive
58,140.0,0.06,0.00,NaN,NaN,494,20,6,LOW,keyword article,transactional,365+,0-30,NaN,NaN,1,0,False negative
64,10.0,0.00,0.00,2808.0,19244.0,106,8,4,LOW,keyword article,informational,91-180,0-30,2000-3500,15000-25000,0,1,False positive


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.